In [17]:
%pip install -q pandas numpy scikit-learn plotly gradio

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300

soil_data = pd.DataFrame({
    "Sample_ID": [f"SOIL-{i:03d}" for i in range(1, n + 1)],
    "Nitrogen": np.random.normal(250, 45, n).clip(80, 400),
    "Phosphorus": np.random.normal(35, 10, n).clip(5, 80),
    "Potassium": np.random.normal(180, 35, n).clip(60, 300),
    "pH": np.random.normal(6.5, 0.8, n).clip(4, 9),
    "Organic_Carbon": np.random.normal(1.2, 0.4, n).clip(0.2, 3),
    "Moisture": np.random.normal(25, 7, n).clip(5, 50),
    "Electrical_Conductivity": np.random.normal(0.8, 0.3, n).clip(0.1, 2.5)
})

soil_data.to_csv("sample_soil_nutrients.csv", index=False)
display(soil_data.head(10))
print("Dataset shape:", soil_data.shape)
print("Sample dataset created successfully!")

,Sample_ID,Nitrogen,Phosphorus,Potassium,pH,Organic_Carbon,Moisture,Electrical_Conductivity
0,SOIL-001,272.352137,26.710050,206.494602,6.794939,1.250090,30.448528,1.370357
1,SOIL-002,243.778106,29.398190,147.724214,6.185329,1.028238,21.141700,0.781802
2,SOIL-003,279.145984,42.472936,210.436207,6.522996,1.248919,19.272608,0.587478
3,SOIL-004,318.536344,41.103703,227.447325,7.522761,1.417319,24.976379,0.345886
4,SOIL-005,239.463098,34.790984,194.470222,6.652879,1.219544,23.808708,0.259058
5,SOIL-006,239.463837,36.173274,245.687853,6.537149,1.216237,21.827404,0.324759
6,SOIL-007,321.064577,47.776649,152.917378,5.412115,0.919203,29.874712,0.880138
7,SOIL-008,284.534563,29.084286,136.437085,7.097003,0.934840,31.687136,0.952618
8,SOIL-009,228.873653,40.470974,117.744791,7.016387,0.638958,25.618848,0.325643
9,SOIL-010,274.415202,32.978073,232.361551,8.230604,1.899831,35.342711,1.068511


Dataset shape: (300, 8)
Sample dataset created successfully!


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features = [
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "pH",
    "Organic_Carbon",
    "Moisture",
    "Electrical_Conductivity"
]

X = soil_data[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
pca_scores = pca.fit_transform(X_scaled)

variance = pca.explained_variance_ratio_ * 100

print("PCA completed!")
print(f"PC1 variance: {variance[0]:.2f}%")
print(f"PC2 variance: {variance[1]:.2f}%")
print(f"Combined PC1 + PC2: {variance[:2].sum():.2f}%")

PCA completed!
PC1 variance: 17.25%
PC2 variance: 16.47%
Combined PC1 + PC2: 33.72%


In [4]:
import plotly.express as px

scores = pd.DataFrame({
    "PC1": pca_scores[:, 0],
    "PC2": pca_scores[:, 1],
    "Sample_ID": soil_data.loc[X.index, "Sample_ID"].values
})

fig = px.scatter(
    scores,
    x="PC1",
    y="PC2",
    hover_name="Sample_ID",
    title="SoilVision AI — PCA Sample Distribution",
    template="plotly_dark"
)

fig.update_traces(marker=dict(size=9, opacity=0.75))
fig.update_layout(height=550)
fig.show()

variance_fig = px.bar(
    x=[f"PC{i+1}" for i in range(len(variance))],
    y=variance,
    labels={"x": "Principal Component", "y": "Explained Variance (%)"},
    title="PCA Explained Variance",
    template="plotly_dark"
)
variance_fig.show()

In [11]:

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from google.colab import files

# Upload your dataset
uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

# Select nutrient columns
nutrient_cols = [
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "pH",
    "Organic_Carbon",
    "Moisture",
    "Electrical_Conductivity"
]

# Check required columns
missing = [col for col in nutrient_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

# Standardize data
X = df[nutrient_cols].apply(pd.to_numeric, errors="coerce")
valid_rows = X.notna().all(axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.loc[valid_rows])

# Apply PCA
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# Add PCA results to dataset
df["PC1"] = float("nan")
df["PC2"] = float("nan")
df.loc[valid_rows, "PC1"] = components[:, 0]
df.loc[valid_rows, "PC2"] = components[:, 1]

# Display results
print("PCA Explained Variance:")
print(f"PC1: {pca.explained_variance_ratio_[0] * 100:.2f}%")
print(f"PC2: {pca.explained_variance_ratio_[1] * 100:.2f}%")

print("\nDataset with PCA results:")
display(df.head(10))

# Download updated dataset
output_file = "soil_nutrients_with_pca.csv"
df.to_csv(output_file, index=False)
files.download(output_file)


Saving sample_soil_nutrients (1).csv to sample_soil_nutrients (1).csv
PCA Explained Variance:
PC1: 17.25%
PC2: 16.47%

Dataset with PCA results:


,Sample_ID,Nitrogen,Phosphorus,Potassium,pH,Organic_Carbon,Moisture,Electrical_Conductivity,PC1,PC2
0,SOIL-001,272.35,26.71,206.49,6.79,1.25,30.45,1.37,0.521428,0.719158
1,SOIL-002,243.78,29.40,147.72,6.19,1.03,21.14,0.78,-0.399634,-0.250513
2,SOIL-003,279.15,42.47,210.44,6.52,1.25,19.27,0.59,-0.273405,0.035444
3,SOIL-004,318.54,41.10,227.45,7.52,1.42,24.98,0.35,1.438614,0.545962
4,SOIL-005,239.46,34.79,194.47,6.65,1.22,23.81,0.26,0.106312,0.064605
5,SOIL-006,239.46,36.17,245.69,6.54,1.22,21.83,0.32,-0.392436,0.508637
6,SOIL-007,321.06,47.78,152.92,5.41,0.92,29.87,0.88,0.447399,-1.553412
7,SOIL-008,284.53,29.08,136.44,7.10,0.93,31.69,0.95,1.531718,-0.642424
8,SOIL-009,228.87,40.47,117.74,7.02,0.64,25.62,0.33,0.577283,-2.083329
9,SOIL-010,274.42,32.98,232.36,8.23,1.90,35.34,1.07,1.760569,1.551064


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:

%pip install -q gradio pandas numpy scikit-learn plotly

import os
import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# =========================================================
# SOILVISION AI PRO - COMPLETE SOIL PCA ANALYTICS WEBSITE
# =========================================================

NUTRIENTS = [
    "Nitrogen", "Phosphorus", "Potassium", "pH",
    "Organic_Carbon", "Moisture", "Electrical_Conductivity"
]

PALETTE = ["#06b6d4", "#8b5cf6", "#10b981", "#f59e0b", "#f43f5e"]

CSS = """
.gradio-container {
    max-width: 1500px !important;
    margin: auto !important;
    font-family: Arial, sans-serif;
}
#hero {
    background: linear-gradient(125deg, #06152e 0%, #103b58 55%, #087e8b 100%);
    color: white;
    padding: 30px;
    border-radius: 20px;
    margin-bottom: 16px;
}
#hero h1 {font-size: 38px; margin-bottom: 8px;}
#hero p {color: #d5f6ff; font-size: 15px;}
#hero span {
    background: #164e63;
    padding: 7px 12px;
    border-radius: 20px;
    font-size: 12px;
}
.metric {
    background: linear-gradient(135deg, #ffffff, #edfaff);
    border: 1px solid #d6eaf1;
    padding: 17px;
    border-radius: 15px;
    min-width: 145px;
    flex: 1;
}
.metric .label {color: #527080; font-size: 13px;}
.metric .value {color: #0b3552; font-size: 27px; font-weight: bold; margin-top: 7px;}
.section-title {color: #0c5268; font-weight: bold;}
"""

# ------------------- ANALYSIS ENGINE ---------------------

def analyze(file):
    if file is None:
        raise gr.Error("Upload a CSV dataset first.")

    try:
        df = pd.read_csv(file)
    except Exception as e:
        raise gr.Error(f"Cannot read CSV file: {e}")

    if df.empty:
        raise gr.Error("The uploaded dataset is empty.")

    missing = [c for c in NUTRIENTS if c not in df.columns]
    if missing:
        raise gr.Error(
            "Required columns missing: " + ", ".join(missing)
        )

    X = df[NUTRIENTS].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Remove rows with no nutrient measurements at all
    valid = X.notna().any(axis=1)
    if valid.sum() < 2:
        raise gr.Error("At least two usable soil samples are required.")

    X_valid = X.loc[valid]

    # Fill individual missing measurements with column medians
    imputer = SimpleImputer(strategy="median")
    X_imputed_array = imputer.fit_transform(X_valid)

    # Standardize features before PCA
    scaler = StandardScaler()
    scaled = scaler.fit_transform(X_imputed_array)

    pca = PCA(n_components=2)
    scores = pca.fit_transform(scaled)
    variance = pca.explained_variance_ratio_ * 100

    # Result dataset retains original rows
    result = df.copy()
    result["PC1"] = np.nan
    result["PC2"] = np.nan
    result["Soil_Cluster"] = "Not analyzed"

    result.loc[valid, "PC1"] = scores[:, 0]
    result.loc[valid, "PC2"] = scores[:, 1]

    # Cluster samples in PCA space where possible
    n_clusters = min(3, len(scores))
    if n_clusters >= 2:
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = km.fit_predict(scores)
        result.loc[valid, "Soil_Cluster"] = [
            f"Cluster {i+1}" for i in labels
        ]

    # Loadings indicate feature relationships to principal components
    loadings = pd.DataFrame(
        pca.components_.T,
        index=NUTRIENTS,
        columns=["PC1", "PC2"]
    )
    loadings.index.name = "Nutrient"

    # Overview metrics
    cards = f"""
    <div style="display:flex;gap:12px;flex-wrap:wrap">
      <div class="metric">
        <div class="label">TOTAL SAMPLES</div>
        <div class="value">{len(df):,}</div>
      </div>
      <div class="metric">
        <div class="label">ANALYZED SAMPLES</div>
        <div class="value">{int(valid.sum()):,}</div>
      </div>
      <div class="metric">
        <div class="label">NUTRIENT FEATURES</div>
        <div class="value">{len(NUTRIENTS)}</div>
      </div>
      <div class="metric">
        <div class="label">PC1 VARIANCE</div>
        <div class="value">{variance[0]:.1f}%</div>
      </div>
      <div class="metric">
        <div class="label">PC1 + PC2</div>
        <div class="value">{variance.sum():.1f}%</div>
      </div>
    </div>
    """

    # PCA scatter plot
    plot_df = result.loc[valid].copy()
    plot_df["Sample"] = plot_df.index.astype(str)

    scatter = px.scatter(
        plot_df,
        x="PC1",
        y="PC2",
        color="Soil_Cluster",
        hover_data=NUTRIENTS,
        title="Soil Sample Map — PCA Projection",
        template="plotly_white",
        color_discrete_sequence=PALETTE
    )
    scatter.update_traces(marker=dict(size=10, opacity=0.85))
    scatter.update_layout(
        height=520,
        xaxis_title=f"PC1 ({variance[0]:.2f}% variance)",
        yaxis_title=f"PC2 ({variance[1]:.2f}% variance)",
        legend_title="Sample cluster"
    )

    # Explained variance chart
    variance_df = pd.DataFrame({
        "Component": ["PC1", "PC2"],
        "Explained Variance (%)": variance
    })

    variance_fig = px.bar(
        variance_df,
        x="Component",
        y="Explained Variance (%)",
        text="Explained Variance (%)",
        color="Component",
        title="Principal Component Explained Variance",
        template="plotly_white",
        color_discrete_sequence=PALETTE
    )
    variance_fig.update_traces(
        texttemplate="%{text:.2f}%",
        textposition="outside"
    )
    variance_fig.update_layout(height=390)

    # Nutrient distribution
    nutrient_long = X_valid.copy()
    nutrient_long.index.name = "Sample"
    nutrient_long = nutrient_long.reset_index().melt(
        id_vars="Sample",
        var_name="Nutrient",
        value_name="Value"
    )

    nutrient_fig = px.box(
        nutrient_long,
        x="Nutrient",
        y="Value",
        color="Nutrient",
        title="Nutrient Distribution Across Samples",
        template="plotly_white",
        color_discrete_sequence=PALETTE
    )
    nutrient_fig.update_layout(height=460, showlegend=False)

    # Correlation heatmap
    corr = X_valid.corr()
    heatmap = px.imshow(
        corr,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title="Nutrient Correlation Matrix",
        template="plotly_white"
    )
    heatmap.update_layout(height=550)

    # PCA feature loading chart
    loadings_long = loadings.reset_index().melt(
        id_vars="Nutrient",
        var_name="Component",
        value_name="Loading"
    )
    loading_fig = px.bar(
        loadings_long,
        x="Nutrient",
        y="Loading",
        color="Component",
        barmode="group",
        title="Nutrient Contributions to PC1 and PC2",
        template="plotly_white",
        color_discrete_sequence=PALETTE
    )
    loading_fig.update_layout(height=460)

    # Summary report
    cluster_counts = result.loc[valid, "Soil_Cluster"].value_counts()
    cluster_text = "\n".join(
        f"- {name}: {count} samples"
        for name, count in cluster_counts.items()
    )

    report = f"""
# SoilVision AI — Analysis Report

## 1. Dataset Summary
- Total dataset rows: **{len(df)}**
- Samples used in PCA: **{int(valid.sum())}**
- Samples with no usable nutrient data: **{int((~valid).sum())}**
- Features analyzed: **{len(NUTRIENTS)}**

Individual missing nutrient values were filled using the median
of each feature before standardization.

## 2. PCA Results
- **PC1 explained variance:** {variance[0]:.2f}%
- **PC2 explained variance:** {variance[1]:.2f}%
- **Combined explained variance:** {variance.sum():.2f}%

PC1 and PC2 summarize variation across the standardized nutrient
features. The combined percentage indicates how much total variation
is represented in this two-dimensional view.

## 3. Exploratory Clustering
K-means was used to group the analyzed samples in PCA coordinate space.
These groups are exploratory patterns, not certified soil-quality grades.

{cluster_text}

## 4. Interpretation Notes
- Review the nutrient correlation heatmap for relationships between features.
- Review the loading chart to understand how features relate to PC1 and PC2.
- Use the PCA scatter plot to explore similarities among samples.
- A high or low PCA score does not, by itself, mean soil is healthy or unhealthy.
- Fertility recommendations require crop type, local soil standards and agronomic context.

## 5. Features
{", ".join(NUTRIENTS)}
"""

    # Export files
    os.makedirs("/content/soilvision_exports", exist_ok=True)
    csv_path = "/content/soilvision_exports/SoilVision_PCA_Results.csv"
    loadings_path = "/content/soilvision_exports/PCA_Feature_Loadings.csv"
    report_path = "/content/soilvision_exports/SoilVision_Report.txt"

    result.to_csv(csv_path, index=False)
    loadings.to_csv(loadings_path)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)

    return (
        cards, report, scatter, variance_fig, nutrient_fig,
        heatmap, loading_fig, result.head(30),
        csv_path, loadings_path, report_path
    )


# ----------------------- WEBSITE UI ----------------------

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="cyan", secondary_hue="blue"),
    css=CSS,
    title="SoilVision AI Pro"
) as demo:

    gr.HTML("""
    <div id="hero">
      <h1>🌱 SoilVision AI Pro</h1>
      <p>Advanced Soil Nutrient Intelligence & Principal Component Analysis</p>
      <span>AI-ASSISTED EXPLORATION</span>
      &nbsp; <span>PCA LABORATORY</span>
      &nbsp; <span>DATA INTELLIGENCE</span>
    </div>
    """)

    gr.Markdown(
        "### Your Soil Analytics Command Center\n"
        "Explore nutrient patterns, reduce dimensionality and inspect "
        "sample groupings through interactive visualizations."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Dataset Workspace")
            file_input = gr.File(
                label="Upload Soil Nutrient CSV",
                file_types=[".csv"],
                type="filepath"
            )
            run_button = gr.Button(
                "🚀 Analyze Dataset", variant="primary", size="lg"
            )
            gr.Markdown(
                "**Required nutrient columns**\n\n"
                + "\n".join(f"- `{c}`" for c in NUTRIENTS)
            )
            gr.Markdown(
                "The dataset should contain numeric nutrient values. "
                "Sample IDs or additional columns may also be present."
            )

        with gr.Column(scale=2):
            gr.Markdown("### 📊 Live Analysis Overview")
            cards_out = gr.HTML(
                "<p>Upload a dataset and click Analyze Dataset to begin.</p>"
            )
            report_out = gr.Markdown()

    with gr.Tabs():
        with gr.Tab("🗺️ PCA Laboratory"):
            gr.Markdown("### Principal Component Analysis")
            scatter_out = gr.Plot(label="PCA Sample Map")
            variance_out = gr.Plot(label="Explained Variance")

        with gr.Tab("🌿 Nutrient Intelligence"):
            gr.Markdown("### Explore Nutrient Distributions")
            nutrient_out = gr.Plot(label="Nutrient Box Plots")
            heatmap_out = gr.Plot(label="Nutrient Correlation Heatmap")

        with gr.Tab("🧪 Feature Loadings"):
            gr.Markdown(
                "Loadings show the relationship between each nutrient "
                "and the two principal components."
            )
            loading_out = gr.Plot(label="PCA Feature Loadings")

        with gr.Tab("📑 Dataset Explorer"):
            table_out = gr.Dataframe(
                label="Analyzed Dataset Preview",
                interactive=False
            )
            gr.Markdown("### Download Your Results")
            csv_out = gr.File(label="PCA Results CSV")
            loading_csv_out = gr.File(label="Feature Loadings CSV")
            report_file_out = gr.File(label="Text Analysis Report")

    run_button.click(
        fn=analyze,
        inputs=[file_input],
        outputs=[
            cards_out, report_out, scatter_out, variance_out,
            nutrient_out, heatmap_out, loading_out, table_out,
            csv_out, loading_csv_out, report_file_out
        ]
    )

    gr.Markdown("""
    ---
    **SoilVision AI Pro** · Soil Nutrient Data Analysis Using PCA

    *Exploratory data analysis only — not a laboratory soil certification
    or a substitute for agronomic advice.*
    """)

demo.launch(share=True, inbrowser=True)


/tmp/ipykernel_1612/590165936.py:315: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0287c7dede3f7b74cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
